# Кольцевые коммуникации


In [1]:
from datetime import timedelta
import socket

import torch
import torch.distributed as dist
import torch.multiprocessing as mp


def ring_reduce_scatter(tensor):
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    chunks = [chunk.clone() for chunk in tensor.chunk(world_size)]
    next_rank = (rank + 1) % world_size
    previous_rank = (rank - 1) % world_size

    for step in range(world_size - 1):
        send_index = (rank - step - 1) % world_size
        receive_index = (rank - step - 2) % world_size
        received = torch.empty_like(chunks[receive_index])
        requests = dist.batch_isend_irecv([
            dist.P2POp(dist.isend, chunks[send_index], next_rank),
            dist.P2POp(dist.irecv, received, previous_rank),
        ])
        for request in requests:
            request.wait()
        chunks[receive_index].add_(received)

    return chunks[rank]


def ring_all_gather(shard):
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    chunks = [torch.empty_like(shard) for _ in range(world_size)]
    chunks[rank].copy_(shard)
    next_rank = (rank + 1) % world_size
    previous_rank = (rank - 1) % world_size

    for step in range(world_size - 1):
        send_index = (rank - step) % world_size
        receive_index = (rank - step - 1) % world_size
        requests = dist.batch_isend_irecv([
            dist.P2POp(dist.isend, chunks[send_index], next_rank),
            dist.P2POp(dist.irecv, chunks[receive_index], previous_rank),
        ])
        for request in requests:
            request.wait()

    return torch.cat(chunks)


def ring_all_reduce(tensor):
    return ring_all_gather(ring_reduce_scatter(tensor))


## Сравнение с PyTorch


In [2]:
def check_ring(rank, world_size, port):
    torch.set_num_threads(1)
    dist.init_process_group(
        'gloo', init_method=f'tcp://127.0.0.1:{port}',
        rank=rank, world_size=world_size, timeout=timedelta(seconds=30),
    )
    try:
        generator = torch.Generator().manual_seed(42 + rank)
        for chunk_size in (1, 7, 31):
            values = torch.randn(world_size * chunk_size, generator=generator, dtype=torch.float64)
            original = values.clone()
            expected_shard = torch.empty(chunk_size, dtype=values.dtype)
            dist.reduce_scatter_tensor(expected_shard, values.clone(), op=dist.ReduceOp.SUM)
            torch.testing.assert_close(ring_reduce_scatter(values), expected_shard)
            torch.testing.assert_close(values, original, rtol=0, atol=0)

            shard = torch.randn(chunk_size, generator=generator, dtype=torch.float64)
            original_shard = shard.clone()
            expected_parts = [torch.empty_like(shard) for _ in range(world_size)]
            dist.all_gather(expected_parts, shard)
            torch.testing.assert_close(ring_all_gather(shard), torch.cat(expected_parts))
            torch.testing.assert_close(shard, original_shard, rtol=0, atol=0)

            expected_sum = values.clone()
            dist.all_reduce(expected_sum, op=dist.ReduceOp.SUM)
            torch.testing.assert_close(ring_all_reduce(values), expected_sum)
            torch.testing.assert_close(values, original, rtol=0, atol=0)

    finally:
        dist.destroy_process_group()


def run_check(world_size):
    with socket.socket() as server:
        server.bind(('127.0.0.1', 0))
        port = server.getsockname()[1]
    mp.start_processes(check_ring, args=(world_size, port), nprocs=world_size,
                       join=True, start_method='fork')
    print(f'OK: все три операции прошли проверки на {world_size} процессах.')


In [3]:
run_check(2)


OK: все три операции прошли проверки на 2 процессах.


In [4]:
run_check(4)


OK: все три операции прошли проверки на 4 процессах.
